# 1. Refinement Loop Prompt Optimization


In [ ]:
import os
from pathlib import Path
from openai import OpenAI
from tinydb import TinyDB

from utils.util import clone_db

from optimization.hypotheses.HypothesesRefiner import HypothesesRefiner
from optimization.policy.PolicyRefiner import PolicyRefiner
from optimization.prompts.FrozenLakePrompts import FrozenLakePrompts
from optimization.PlaybookRefactorer import PlaybookRefactorer

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key= os.getenv("OPENROUTER_API_KEY")
)
models = ["x-ai/grok-4.1-fast"]

## 1.1 PolicyRefiner Prompt Optimizer

In [ ]:
# Identify wrong strategies
# Strategies
## Suboptimal strategy (e.g. deadlock back and forth movement)
### Neveer get closer to an Hole H (deadlock)
### Always go to nearest hole H (death)

## Contradictions
### Always go to H vs avoid H (go to hole)

## Non-deterministic movements
### Trajectory shows sometims navigator failes to move as expected (want to see strategies that emphazise that movement sometimes not as expected)

BASE_IN = Path("../store/ablation/policyrefiner")
for i in range(len(models)):
    BASE_OUT = BASE_IN / f"{models[i].split('/')[-1]}"
    hypo = TinyDB(BASE_IN / "base_hypotheses.json")

    src_policy_contra_hole = BASE_IN / "contra_hole.json"
    src_policy_sub_avoidhole = BASE_IN / "sub_avoidhole.json"
    src_policy_sub_gohole = BASE_IN / "sub_gohole.json"
    src_nondet_movements = BASE_IN / "nondeterministic.json"

    pb_contra_hole = clone_db(src_policy_contra_hole, BASE_OUT / "contra_hole.json")
    pb_sub_avoidhole = clone_db(src_policy_sub_avoidhole, BASE_OUT / "sub_avoidhole.json")
    pb_sub_gohole = clone_db(src_policy_sub_gohole, BASE_OUT / "sub_gohole.json")
    pb_nondet_movements = clone_db(src_nondet_movements, BASE_OUT / "nondeterministic.json")

    str_contra_hole = open(BASE_IN / "contra_hole.txt").read()
    str_sub_avoidhole = open(BASE_IN / "sub_avoidhole.txt").read()
    str_sub_gohole = open(BASE_IN / "sub_gohole.txt").read()
    str_nondet_movements = open(BASE_IN / "nondeterministic.txt").read()

    testcases = [
        (pb_contra_hole, str_contra_hole),
        (pb_sub_avoidhole, str_sub_avoidhole),
        (pb_sub_gohole, str_sub_gohole),
        (pb_nondet_movements, str_nondet_movements)
    ]

    for j in range(len(testcases)):
        pb, traj = testcases[j]
        prompts = FrozenLakePrompts(pb, hypo)
        policy_refiner = PolicyRefiner(client, models[i], pb, prompts)
        policy_refiner.run(traj, debug=False)

## 1.2 HypothesesRefiner Prompt Optimizer

In [ ]:
# Identify wrong hypotheses
# Hypotheses Refinement
## Falsified hypotheses (e.g. trajectory shows hypothese is cleary wrong, e.g. must be removed/modified)
### Hole can teleport
### Entering F yields reward of 0.5
### move_left = move_right, move_up = move_down

## Unfalsified hypotheses (e.g. trajectory shows hypothese is not falsified, e.g. it still holds)
### Normal trajectory + normal hypothesis

## New hypotheses (e.g. trajectory shows new dynamics, e.g. make new hypotheses)
### Hole terminates
### Move over the edge

## Existing Conflict identified by playbook refactorer (trajectory shows one must be true, so the refinment should only choose this)
### Conflict between h telport terminates
### Conflict move_left = move_right, move_up = move_down
BASE_IN = Path("../store/ablation/hypothesesrefiner")
for i in range(len(models)):
    BASE_OUT = BASE_IN / f"{models[i].split('/')[-1]}"

    src_hypotheses_contra_hole = BASE_IN / "contra_hole.json"
    src_hypotheses_contra_move = BASE_IN / "contra_move.json"
    src_hypotheses_fals_hole = BASE_IN / "fals_hole.json"
    src_hypotheses_fals_move = BASE_IN / "fals_move.json"
    src_hypotheses_fals_reward = BASE_IN / "fals_reward.json"
    src_hypotheses_new_edge = BASE_IN / "new_edge.json"
    src_hypotheses_new_hole = BASE_IN / "new_hole.json"
    src_hypotheses_true = BASE_IN / "true_hypotheses.json"

    pb_contra_hole = clone_db(src_hypotheses_contra_hole, BASE_OUT / "contra_hole.json")
    pb_contra_move = clone_db(src_hypotheses_contra_move, BASE_OUT / "contra_move.json")
    pb_fals_hole = clone_db(src_hypotheses_fals_hole, BASE_OUT / "fals_hole.json")
    pb_fals_move = clone_db(src_hypotheses_fals_move, BASE_OUT / "fals_move.json")
    pb_fals_reward = clone_db(src_hypotheses_fals_reward, BASE_OUT / "fals_reward.json")
    pb_new_edge = clone_db(src_hypotheses_new_edge, BASE_OUT / "new_edge.json")
    pb_new_hole = clone_db(src_hypotheses_new_hole, BASE_OUT / "new_hole.json")
    pb_true = clone_db(src_hypotheses_true, BASE_OUT / "true_hypotheses.json")

    str_contra_hole = open(BASE_IN / "contra_hole.txt").read()
    str_contra_move = open(BASE_IN / "contra_move.txt").read()
    str_fals_hole = open(BASE_IN / "fals_hole.txt").read()
    str_fals_move = open(BASE_IN / "fals_move.txt").read()
    str_fals_reward = open(BASE_IN / "fals_reward.txt").read()
    str_new_edge = open(BASE_IN / "new_edge.txt").read()
    str_new_hole = open(BASE_IN / "new_hole.txt").read()
    str_true = open(BASE_IN / "true_trajectory.txt").read()

    playbooks = [
        pb_contra_hole,
        pb_contra_move,
        pb_fals_hole,
        pb_fals_move,
        pb_fals_reward,
        pb_new_edge,
        pb_new_hole,
        pb_true
    ]
    trajectories = [
        str_contra_hole,
        str_contra_move,
        str_fals_hole,
        str_fals_move,
        str_fals_reward,
        str_new_edge,
        str_new_hole,
        str_true
    ]

    for j in range(len(playbooks)):
        hr = HypothesesRefiner(client, models[i], playbooks[j])
        hr.run(trajectories[j], debug=True)


## 1.3 Playbook Refactorer Prompt Optimizer

In [ ]:
BASE_IN = Path("../store/ablation/playbookrefactorer")
# Duplicate Information
## Strategies 3x
## Hypotheses 3x
print("# DUPLICATE ABLATION #")
for i in range(len(models)):
    BASE_OUT = Path(f"../store/ablation/playbookrefactorer/{models[i].split('/')[-1]}")
    BASE_OUT.mkdir(parents=True, exist_ok=True)
    for j in range(3):
        src_policies = BASE_IN / f"dup_{j}_policies.json"
        src_hypotheses = BASE_IN / f"dup_{j}_hypotheses.json"

        policies = clone_db(src_policies, BASE_OUT / f"dup_{j}_policies.json")
        hypotheses = clone_db(src_hypotheses, BASE_OUT / f"dup_{j}_hypotheses.json")

        ref_strat = PlaybookRefactorer(client, models[i], policies)
        ref_hyp = PlaybookRefactorer(client, models[i], hypotheses)

        print(f"## Strategy Duplicate: {j} ##")
        ref_strat.run(debug=True)

        print(f"## Hypothesis Duplicate: {j} ##")
        ref_hyp.run(debug=True)

        policies.close()
        hypotheses.close()


## Non-atomic entries (multiple informations)
### Strategies 3x
### Hypotheses 3x
#print("# NON-ATOMIC ABLATION #")
#for i in range(len(models)):
#    BASE_OUT = Path(f"../store/ablation/playbookrefactorer/{models[i].split('/')[-1]}")
#    BASE_OUT.mkdir(parents=True, exist_ok=True)
#    for j in range(3):
#        src_policies = BASE_IN / f"atom_{j}_policies.json"
#        src_hypotheses = BASE_IN / f"atom_{j}_hypotheses.json"
#
#        policies = clone_db(src_policies, BASE_OUT / f"atom_{j}_policies.json")
#        hypotheses = clone_db(src_hypotheses, BASE_OUT / f"atom_{j}_hypotheses.json")
#
#        ref_strat = PlaybookRefactorer(client, models[i], policies)
#        
#        ref_hyp = PlaybookRefactorer(client, models[i], hypotheses)
#        print(f"## Strategy Non-Atomic: {j} ##")
#        
#        ref_strat.run(debug=True)
#        print(f"## Hypothesis Non-Atomic: {j} ##")
#        ref_hyp.run(debug=True)
#
#        policies.close()
#        hypotheses.close()

# Contradicting Entries
## Strategies 3x
## Hypotheses 3x
print("# CONTRADICTING ABLATION #")
for i in range(len(models)):
    BASE_OUT = Path(f"../store/ablation/playbookrefactorer/{models[i].split('/')[-1]}")
    BASE_OUT.mkdir(parents=True, exist_ok=True)
    for j in range(3):
        src_policies = BASE_IN / f"contra_{j}_policies.json"
        src_hypotheses = BASE_IN / f"contra_{j}_hypotheses.json"

        policies = clone_db(src_policies, BASE_OUT / f"contra_{j}_policies.json")
        hypotheses = clone_db(src_hypotheses, BASE_OUT / f"contra_{j}_hypotheses.json")

        ref_strat = PlaybookRefactorer(client, models[i], policies)
        ref_hyp = PlaybookRefactorer(client, models[i], hypotheses)

        print(f"## Strategy Contradicting: {j} ##")
        ref_strat.run(debug=True)

        print(f"## Hypothesis Contradicting: {j} ##")
        ref_hyp.run(debug=True)

        policies.close()
        hypotheses.close()